In [ ]:
using Pkg
using Random
using Statistics
using Printf
using LinearAlgebra
using Logging

function find_project_root(start::AbstractString=pwd())
    active_project = Base.active_project()
    if !isnothing(active_project)
        active_root = dirname(active_project)
        if isfile(joinpath(active_root, "Project.toml")) && isfile(joinpath(active_root, "src", "System1D.jl"))
            return active_root
        end
    end

    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()
Pkg.activate(PROJECT_ROOT; io=devnull)

using Revise
using Plots

includet(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "fermion_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

includet(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model, VMC Warm Start, and GFMC Parameters

This notebook runs fixed-node GFMC for the built-in `SpinlessFermionRing1D` model, but the GFMC walker ensemble is prepared with
`run_gfmc_with_vmc_init(model, gfmc_params, init_configs, vmc_params; ...)`.

The model represents `N` spinless fermions on a periodic ring with length `L = M * a` and lattice potential
`V(x) = V0 * cos(2*pi*x/a)` applied to each particle coordinate.

The VMC warm-start stage samples the model's built-in trial wavefunction before the projector run starts, so the recorded GFMC step `0`
snapshot is already a VMC-equilibrated walker cloud.

Parameters used below:
- Fermion count `N = 3`
- Number of lattice periods `M = 3`
- Lattice spacing `a = 1.0`
- Ring length `L = 3.0`
- Lattice amplitude `V0 = 1.0`
- Twist angle `twist = 0.0`
- VMC warm-start step size `vmc_dt = 2.0e-2`
- VMC warm-start steps `vmc_nsteps = 150`
- GFMC step size `gfmc_dt = 3.0e-3`
- GFMC steps `gfmc_nsteps = 1200`
- GFMC equilibration steps `gfmc_nequil = 200`
- Target population `targetN = 600`
- Feedback strength `feedback = 0.1`
- Reconfiguration interval `reconfiguration_interval = 2`
- Branch-weight cap `branch_cap = 5.0`
- ET averaging window `energy_window = 40`

Trial / node structure:
- The trial state comes from `trial_wavefunction(model)`
- The warm-start VMC stage uses `DriftGaussianProposal()`
- The GFMC stage runs with `use_guiding = true`
- The node policy is `FixedNode()`
- The reconfiguration policy is `SystematicReconfiguration()`

Additional model and VMC-init diagnostics:
- The final plotting cell includes the external cosine potential over the ring.
- It also plots the one-body lattice factor `exp(lambda cos(2*pi*x/a))` and the pair nodal magnitude `|sin(pi r / L)|` that appear inside the many-body trial ansatz.
- The warm-start density panel compares the recorded GFMC step-0 density, which is the post-VMC ensemble, against those trial ingredients.
- When `N == 1`, the notebook also constructs a finite-difference reference by discretizing the single-particle periodic Hamiltonian on a uniform grid, using the standard second-order central-difference Laplacian with periodic wraparound, diagonalizing the resulting symmetric matrix, and normalizing the lowest-eigenvector density `|psi0|^2`.
- When `N > 1`, no exact many-body reference is plotted because this notebook does not include a many-body exact diagonalization solver.


## Julia Construction

The next cell constructs the fermion-ring model, builds both the VMC and GFMC parameter sets, defines the output toggles, and prepares the raw initial configurations.

Edit that cell if you want a different ring size, twist, warm-start length, debug cadence, or CSV output name.


In [ ]:
N = 1
M = 3
a = 1.0
L = M * a
V0 = 1.0
twist = 0.0

model = SpinlessFermionRing1D(N, a, L, V0; twist=twist, D=0.5, node_tol=1.0e-7, trig_eps=1.0e-10)
H = hamiltonian(model)
trial = trial_wavefunction(model)

targetN = 600

vmc_dt = 2.0e-2
vmc_nsteps = 150
vmc_ET0 = -0.2
vmc_params = VMCParams(; dt=vmc_dt, nsteps=vmc_nsteps, targetN=targetN, ET0=vmc_ET0)

gfmc_dt = 3.0e-3
gfmc_nsteps = 1200
gfmc_nequil = 200
gfmc_ET0 = -0.2
feedback = 0.1
reconfiguration_interval = 2
branch_cap = 5.0
energy_window = 40
gfmc_params = GFMCParams(gfmc_dt, gfmc_nsteps, gfmc_nequil, targetN, gfmc_ET0, feedback, reconfiguration_interval, branch_cap, energy_window)

rng_init = MersenneTwister(1234)
initial_positions = sample_uniform_configurations(model, targetN, rng_init)

MODEL_GRID_POINTS = 600
FD_GRID_POINTS = 500
xgrid_model = Float64[i * (L / MODEL_GRID_POINTS) for i in 0:(MODEL_GRID_POINTS - 1)]
rgrid_pair = collect(range(0.0, 0.5 * L; length=MODEL_GRID_POINTS))

function normalized_trial_onebody_density(xgrid::AbstractVector{<:Real}, model::SpinlessFermionRing1D)
    rho = Float64[exp(2 * model.lambda * cos(model.k_lat * x)) for x in xgrid]
    dx = model.L / length(xgrid)
    norm = sum(rho) * dx
    norm > 0 || throw(ArgumentError("Trial density normalization failed (non-positive norm)."))
    rho ./= norm
    return rho
end

function finite_difference_reference_single_particle(L::Real, a::Real, V0::Real, ngrid::Int)
    ngrid >= 8 || throw(ArgumentError("ngrid must be >= 8"))
    dx = L / ngrid
    x = collect(0:(ngrid - 1)) .* dx
    k_lat = 2pi / a

    Hfd = zeros(Float64, ngrid, ngrid)
    kin_diag = 1.0 / dx^2
    kin_off = -0.5 / dx^2
    for i in 1:ngrid
        Hfd[i, i] = kin_diag + V0 * cos(k_lat * x[i])
    end
    for i in 1:(ngrid - 1)
        Hfd[i, i + 1] = kin_off
        Hfd[i + 1, i] = kin_off
    end
    Hfd[1, ngrid] = kin_off
    Hfd[ngrid, 1] = kin_off

    evals, evecs = eigen(Symmetric(Hfd))
    E0 = evals[1]
    psi0 = evecs[:, 1]
    rho0 = abs2.(psi0)
    rho0 ./= (sum(rho0) * dx)
    return x, E0, rho0
end

onebody_potential_curve = Float64[V0 * cos(model.k_lat * x) for x in xgrid_model]
onebody_trial_logamp = Float64[model.lambda * cos(model.k_lat * x) for x in xgrid_model]
onebody_trial_amp = exp.(onebody_trial_logamp)
onebody_trial_density = normalized_trial_onebody_density(xgrid_model, model)
pair_nodal_magnitude = Float64[abs(sin(model.alpha_pair * r)) for r in rgrid_pair]
HAS_PAIR_OBSERVABLES = N >= 2

fd_ref = nothing
if N == 1
    x_fd, E0_fd, rho_fd = finite_difference_reference_single_particle(L, a, V0, FD_GRID_POINTS)
    fd_ref = (x=x_fd, E0=E0_fd, rho=rho_fd)
end

SNAPSHOT_STEPS = nb_default_snapshot_steps(gfmc_nsteps)
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.10 * a
PAIR_SEP_BINS = 80
PERIOD_MARKERS = collect(0.0:a:L)

RUN_LABEL = "VMC warm-started guided fixed node"
RUN_COLOR = :navy
PLOT_TITLE = "Spinless fermion ring GFMC"
MODEL_TRIAL_TITLE = "Model and VMC warm-start diagnostics"
DENSITY_TITLE = "Spinless fermion ring GFMC: pooled one-body densities"
UNPOOLED_DENSITY_TITLE = "Spinless fermion ring GFMC: final per-particle one-body densities"
PAIR_TITLE = HAS_PAIR_OBSERVABLES ?
    "Spinless fermion ring GFMC: pair-separation density" :
    "Spinless fermion ring GFMC: pair-separation density (not defined for N = 1)"
PARTICLE_COLORS = [:navy, :darkorange, :forestgreen, :crimson, :purple, :goldenrod, :deeppink, :teal]

VMC_PROPOSAL = DriftGaussianProposal()
USE_GUIDING = true
NODEPOLICY = FixedNode()
RECONFIGURATION = SystematicReconfiguration()

VMC_SHOW_PROGRESS = false
VMC_PROGRESS_EVERY = 0
VMC_DEBUG_MODE = false
VMC_DEBUG_EVERY = 10

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20

WRITE_RUN_CSV = false
CSV_FILENAME = "spinless_fermion_ring_gfmc_vmc_init.csv"
SAVE_FIGURES = false
FIGURE_STEM = "spinless_fermion_ring_gfmc_vmc_init"


In [ ]:
sim = run_gfmc_with_vmc_init(
    model,
    gfmc_params,
    initial_positions,
    vmc_params;
    vmc_rng=MersenneTwister(41),
    gfmc_rng=MersenneTwister(52),
    proposal=VMC_PROPOSAL,
    use_guiding=USE_GUIDING,
    nodepolicy=NODEPOLICY,
    reconfiguration=RECONFIGURATION,
    vmc_show_progress=VMC_SHOW_PROGRESS,
    vmc_progress_every=VMC_PROGRESS_EVERY,
    vmc_progress_label="VMC warm start",
    vmc_debug=VMC_DEBUG_MODE,
    vmc_debug_every=VMC_DEBUG_EVERY,
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(gfmc_params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

final_snapshot = nb_last_snapshot(sim)
final_pair_sep = Float64[]
for R in final_snapshot
    for i in 1:(length(R) - 1)
        for j in (i + 1):length(R)
            push!(final_pair_sep, distance_1d(model.bc, R[i], R[j]))
        end
    end
end

println("GFMC step 0 corresponds to the VMC warm-start ensemble.")
println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, gfmc_params.nequil, mean_energy, sem_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))
if HAS_PAIR_OBSERVABLES
    println(@sprintf("minimum final pair separation = %.6f", minimum(final_pair_sep)))
    println("count with r < model.node_tol = ", count(r -> r < model.node_tol, final_pair_sep))
else
    println("single-particle run: no pair separations to report.")
end
if fd_ref !== nothing
    println(@sprintf("N = 1 finite-difference reference energy = %.8f", fd_ref.E0))
else
    println("No exact many-body reference plotted: the extra model/trial curves are trial ingredients, not an exact N > 1 solution.")
end

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

function ring_particle_coordinates(snapshot, particle_idx::Integer)
    idx = Int(particle_idx)
    return Float64[R[idx] for R in snapshot]
end

function pair_separations(snapshot)
    rs = Float64[]
    for R in snapshot
        for i in 1:(length(R) - 1)
            for j in (i + 1):length(R)
                push!(rs, distance_1d(model.bc, R[i], R[j]))
            end
        end
    end
    return rs
end

step0_snapshot = sim.walker_positions_history[1]
step0_xs = pooled_ring_coordinates(step0_snapshot)
step0_centers, step0_density = nb_periodic_kde_curve(
    step0_xs;
    xmin=0.0,
    xmax=L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

final_snapshot = nb_last_snapshot(sim)
final_xs = pooled_ring_coordinates(final_snapshot)
final_centers, final_density = nb_periodic_kde_curve(
    final_xs;
    xmin=0.0,
    xmax=L,
    grid_points=DENSITY_GRID_POINTS,
    bandwidth=DENSITY_BANDWIDTH,
)

p_potential = plot(
    xgrid_model,
    onebody_potential_curve;
    xlabel="x",
    ylabel="V(x)",
    title="External cosine potential",
    color=:black,
    linewidth=2.4,
    label="V0 cos(2πx/a)",
    xlims=(0.0, L),
)
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_potential, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

p_trial_onebody = plot(
    xgrid_model,
    onebody_trial_amp;
    xlabel="x",
    ylabel="amplitude",
    title="One-body lattice factor in ψ_T",
    color=:teal,
    linewidth=2.4,
    label="exp(λ cos(2πx/a))",
    xlims=(0.0, L),
)
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_trial_onebody, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

p_pair_factor = plot(
    xlabel="r",
    ylabel="magnitude",
    title=(HAS_PAIR_OBSERVABLES ? "Pair nodal factor magnitude" : "Trivial pair factor for N = 1"),
    xlims=(0.0, 0.5 * L),
)
if HAS_PAIR_OBSERVABLES
    plot!(p_pair_factor, rgrid_pair, pair_nodal_magnitude; color=:darkorange, linewidth=2.4, label="|sin(πr/L)|")
else
    plot!(p_pair_factor, rgrid_pair, ones(length(rgrid_pair)); color=:darkorange, linewidth=2.4, linestyle=:dash, label="identity factor")
    ylims!(p_pair_factor, (0.0, 1.1))
end

warm_start_title = fd_ref === nothing ?
    "VMC step-0 density vs trial ingredients" :
    "VMC step-0 density with N = 1 FD reference"
p_warm_start = plot(
    xlabel="x",
    ylabel="density",
    title=warm_start_title,
    legend=:topright,
    xlims=(0.0, L),
)
plot!(p_warm_start, step0_centers, step0_density; color=RUN_COLOR, linewidth=2.4, label="step 0 density (after VMC)")
plot!(p_warm_start, final_centers, final_density; color=:crimson, linewidth=2.2, linestyle=:dot, label="final GFMC density")
plot!(p_warm_start, xgrid_model, onebody_trial_density; color=:gray30, linewidth=2.2, linestyle=:dash, label=(N == 1 ? "trial |ψ_T|^2" : "one-body lattice factor only"))
if fd_ref !== nothing
    plot!(p_warm_start, fd_ref.x, fd_ref.rho; color=:black, linewidth=2.2, linestyle=:dashdot, label="FD |ψ0|^2")
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(p_warm_start, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

model_trial_fig = plot(p_potential, p_trial_onebody, p_pair_factor, p_warm_start; layout=(2, 2), size=(1400, 900), plot_title=MODEL_TRIAL_TITLE)
display(model_trial_fig)
nb_save_figure(model_trial_fig, PATHS.figures_dir, FIGURE_STEM, "model_trial"; enabled=SAVE_FIGURES)

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="pooled one-body density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    step_label = step_idx == 0 ? "step 0 (after VMC warm start)" : "step $(step_idx)"
    plot!(density_fig, centers, density; label=step_label, color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)

particle_colors = [PARTICLE_COLORS[1 + mod(i - 1, length(PARTICLE_COLORS))] for i in 1:model.N]
unpooled_density_fig = plot(
    xlabel="x",
    ylabel="one-body density",
    title=UNPOOLED_DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for particle_idx in 1:model.N
    xs = ring_particle_coordinates(final_snapshot, particle_idx)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(unpooled_density_fig, centers, density; label="particle $(particle_idx)", color=particle_colors[particle_idx], linewidth=2.4)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(unpooled_density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(unpooled_density_fig)
nb_save_figure(unpooled_density_fig, PATHS.figures_dir, FIGURE_STEM, "density_unpooled"; enabled=SAVE_FIGURES)

final_pair_sep = pair_separations(final_snapshot)
if HAS_PAIR_OBSERVABLES
    centers, density = nb_density_curve(
        final_pair_sep;
        nbins=PAIR_SEP_BINS,
        xmin=0.0,
        xmax=0.5 * L,
        smoothing_window=9,
    )
    pair_fig = plot(
        centers,
        density;
        xlabel="r",
        ylabel="density",
        title=PAIR_TITLE,
        linewidth=2.4,
        color=:darkorange,
        label=false,
    )
else
    pair_fig = plot(
        xlabel="r",
        ylabel="density",
        title=PAIR_TITLE,
        legend=false,
        xlims=(0.0, 0.5 * L),
        ylims=(0.0, 1.0),
    )
    annotate!(pair_fig, 0.25 * L, 0.5, Plots.text("No pair-separation density for N = 1", 11, :darkorange))
end
display(pair_fig)
nb_save_figure(pair_fig, PATHS.figures_dir, FIGURE_STEM, "pair_density"; enabled=SAVE_FIGURES)
